In [1]:
#!/usr/bin/env python3
"""
PETSc TAO Optimization Example
==============================

This script demonstrates how to use PETSc TAO (Toolkit for Advanced Optimization) 
for solving optimization problems in Python.

Installation Requirements:
--------------------------
pip install petsc4py
# or with conda:
# conda install -c conda-forge petsc4py

For more complex installations, you might need to install PETSc first:
# conda install -c conda-forge petsc
"""

import numpy as np
import petsc4py
petsc4py.init()
from petsc4py import PETSc
import time


In [2]:


class OptimizationProblem:
    """Base class for optimization problems using PETSc TAO"""
    
    def __init__(self, n_vars):
        self.n_vars = n_vars
        self.tao = None
        self.x = None
        self.f = None
        self.g = None
        
    def setup_vectors(self):
        """Initialize PETSc vectors for variables, objective, and gradient"""
        self.x = PETSc.Vec().createSeq(self.n_vars)
        self.f = PETSc.Vec().createSeq(1)
        self.g = PETSc.Vec().createSeq(self.n_vars)
        
    def setup_tao(self, algorithm='nls'):
        """
        Setup TAO solver
        
        Available algorithms:
        - 'nls': Newton Line Search
        - 'lmvm': Limited Memory Variable Metric
        - 'cg': Conjugate Gradient
        - 'nm': Nelder-Mead
        - 'bqpip': Bounded Quasi-Newton Interior Point
        """
        self.tao = PETSc.TAO().create()
        self.tao.setType(algorithm)
        
        # Set the objective and gradient functions
        self.tao.setObjective(self.objective_function)
        self.tao.setGradient(self.gradient_function)
        
        # Set initial guess
        self.tao.setInitialVector(self.x)
        
        return self.tao
    
    def objective_function(self, tao, x, f):
        """Objective function callback - to be implemented by subclasses"""
        raise NotImplementedError
        
    def gradient_function(self, tao, x, g):
        """Gradient function callback - to be implemented by subclasses"""
        raise NotImplementedError
    
    def solve(self, max_iter=1000, tolerance=1e-6):
        """Solve the optimization problem"""
        if self.tao is None:
            raise ValueError("TAO solver not initialized. Call setup_tao() first.")
            
        # Set convergence tolerances
        self.tao.setTolerances(fatol=tolerance, frtol=tolerance)
        self.tao.setMaximumIterations(max_iter)
        
        # Solve the problem
        start_time = time.time()
        self.tao.solve()
        solve_time = time.time() - start_time
        
        # Get results
        solution = self.tao.getSolution()
        objective_value = self.tao.getObjectiveValue()
        iterations = self.tao.getIterationNumber()
        convergence_reason = self.tao.getConvergedReason()
        
        return {
            'solution': solution.getArray().copy(),
            'objective_value': objective_value,
            'iterations': iterations,
            'convergence_reason': convergence_reason,
            'solve_time': solve_time
        }
    
    def cleanup(self):
        """Clean up PETSc objects"""
        if self.x: self.x.destroy()
        if self.f: self.f.destroy()
        if self.g: self.g.destroy()
        if self.tao: self.tao.destroy()


class RosenbrockProblem(OptimizationProblem):
    """
    Classic Rosenbrock function optimization problem
    
    f(x,y) = (a-x)² + b(y-x²)²
    
    Global minimum at (a,a²) with value 0
    Default: a=1, b=100, minimum at (1,1)
    """
    
    def __init__(self, a=1.0, b=100.0):
        super().__init__(n_vars=2)
        self.a = a
        self.b = b
        self.setup_vectors()
        
        # Set initial guess (can be changed)
        self.x.setValues([0, 1], [-1.2, 1.0])
        self.x.assemblyBegin()
        self.x.assemblyEnd()
    
    def objective_function(self, tao, x, f):
        """Compute Rosenbrock objective function"""
        x_array = x.getArray()
        x1, x2 = x_array[0], x_array[1]
        
        # f(x,y) = (a-x)² + b(y-x²)²
        obj_val = (self.a - x1)**2 + self.b * (x2 - x1**2)**2
        
        f.setValue(0, obj_val)
        f.assemblyBegin()
        f.assemblyEnd()
    
    def gradient_function(self, tao, x, g):
        """Compute Rosenbrock gradient"""
        x_array = x.getArray()
        x1, x2 = x_array[0], x_array[1]
        
        # ∂f/∂x = -2(a-x) - 4bx(y-x²)
        # ∂f/∂y = 2b(y-x²)
        grad_x1 = -2*(self.a - x1) - 4*self.b*x1*(x2 - x1**2)
        grad_x2 = 2*self.b*(x2 - x1**2)
        
        g.setValues([0, 1], [grad_x1, grad_x2])
        g.assemblyBegin()
        g.assemblyEnd()


class QuadraticProblem(OptimizationProblem):
    """
    Simple quadratic optimization problem
    
    f(x) = ½xᵀQx + cᵀx + d
    
    Where Q is positive definite, c is linear term, d is constant
    """
    
    def __init__(self, Q, c, d=0.0):
        self.Q = np.array(Q)
        self.c = np.array(c)
        self.d = d
        n = len(c)
        
        super().__init__(n_vars=n)
        self.setup_vectors()
        
        # Set initial guess to zero
        self.x.set(0.0)
        
    def objective_function(self, tao, x, f):
        """Compute quadratic objective function"""
        x_array = x.getArray()
        
        # f(x) = ½xᵀQx + cᵀx + d
        obj_val = 0.5 * np.dot(x_array, np.dot(self.Q, x_array)) + np.dot(self.c, x_array) + self.d
        
        f.setValue(0, obj_val)
        f.assemblyBegin()
        f.assemblyEnd()
    
    def gradient_function(self, tao, x, g):
        """Compute quadratic gradient"""
        x_array = x.getArray()
        
        # ∇f(x) = Qx + c
        grad = np.dot(self.Q, x_array) + self.c
        
        for i in range(len(grad)):
            g.setValue(i, grad[i])
        g.assemblyBegin()
        g.assemblyEnd()


def run_rosenbrock_example():
    """Run Rosenbrock function optimization example"""
    print("="*60)
    print("ROSENBROCK FUNCTION OPTIMIZATION")
    print("="*60)
    print("Problem: minimize f(x,y) = (1-x)² + 100(y-x²)²")
    print("Global minimum: (1, 1) with value 0")
    print()
    
    # Create and solve problem with different algorithms
    algorithms = ['nls', 'lmvm', 'cg']
    
    for alg in algorithms:
        print(f"\nTesting algorithm: {alg.upper()}")
        print("-" * 40)
        
        try:
            problem = RosenbrockProblem()
            problem.setup_tao(algorithm=alg)
            
            result = problem.solve(max_iter=1000, tolerance=1e-8)
            
            print(f"Solution: ({result['solution'][0]:.6f}, {result['solution'][1]:.6f})")
            print(f"Objective value: {result['objective_value']:.2e}")
            print(f"Iterations: {result['iterations']}")
            print(f"Solve time: {result['solve_time']:.4f} seconds")
            print(f"Convergence: {result['convergence_reason']}")
            
            # Check if solution is close to global minimum
            error = np.linalg.norm(result['solution'] - np.array([1.0, 1.0]))
            print(f"Distance from global minimum: {error:.2e}")
            
            problem.cleanup()
            
        except Exception as e:
            print(f"Error with algorithm {alg}: {e}")


def run_quadratic_example():
    """Run quadratic function optimization example"""
    print("\n" + "="*60)
    print("QUADRATIC FUNCTION OPTIMIZATION")
    print("="*60)
    
    # Create a simple 3D quadratic problem
    # f(x) = ½xᵀQx + cᵀx where Q is positive definite
    Q = np.array([[2.0, 0.5, 0.0],
                  [0.5, 3.0, 0.1],
                  [0.0, 0.1, 1.0]])
    c = np.array([1.0, -2.0, 0.5])
    
    print("Problem: minimize f(x) = ½xᵀQx + cᵀx")
    print("Q =", Q)
    print("c =", c)
    
    # Analytical solution: x* = -Q⁻¹c
    analytical_solution = -np.linalg.solve(Q, c)
    print(f"Analytical solution: {analytical_solution}")
    print()
    
    problem = QuadraticProblem(Q, c)
    problem.setup_tao(algorithm='nls')
    
    result = problem.solve()
    
    print("Numerical solution:")
    print(f"Solution: {result['solution']}")
    print(f"Objective value: {result['objective_value']:.2e}")
    print(f"Iterations: {result['iterations']}")
    print(f"Solve time: {result['solve_time']:.4f} seconds")
    
    # Compare with analytical solution
    error = np.linalg.norm(result['solution'] - analytical_solution)
    print(f"Error vs analytical: {error:.2e}")
    
    problem.cleanup()


def main():
    """Main function to run all examples"""
    print("PETSc TAO Optimization Examples")
    print("==============================")
    
    try:
        run_rosenbrock_example()
        run_quadratic_example()
        
        print("\n" + "="*60)
        print("All examples completed successfully!")
        
    except ImportError as e:
        print("Error: PETSc4py not found!")
        print("Please install with: pip install petsc4py")
        print("Or with conda: conda install -c conda-forge petsc4py")
        
    except Exception as e:
        print(f"Error: {e}")
        print("Make sure PETSc is properly installed and configured.")


if __name__ == "__main__":
    main()

PETSc TAO Optimization Examples
ROSENBROCK FUNCTION OPTIMIZATION
Problem: minimize f(x,y) = (1-x)² + 100(y-x²)²
Global minimum: (1, 1) with value 0


Testing algorithm: NLS
----------------------------------------
Error with algorithm nls: 'petsc4py.PETSc.TAO' object has no attribute 'setInitialVector'

Testing algorithm: LMVM
----------------------------------------
Error with algorithm lmvm: 'petsc4py.PETSc.TAO' object has no attribute 'setInitialVector'

Testing algorithm: CG
----------------------------------------
Error with algorithm cg: 'petsc4py.PETSc.TAO' object has no attribute 'setInitialVector'

QUADRATIC FUNCTION OPTIMIZATION
Problem: minimize f(x) = ½xᵀQx + cᵀx
Q = [[2.  0.5 0. ]
 [0.5 3.  0.1]
 [0.  0.1 1. ]]
c = [ 1.  -2.   0.5]
Analytical solution: [-0.70069808  0.80279232 -0.58027923]

Error: 'petsc4py.PETSc.TAO' object has no attribute 'setInitialVector'
Make sure PETSc is properly installed and configured.
